-----

# Documentación del Motor del tecnologo informático 2025

Este motor ha sido diseñado siguiendo principios de **separación de conceptos** y **reutilización de código**. La idea central es que el `engine` contenga toda la lógica genérica y repetitiva, mientras que la carpeta de tu juego (`arkanoid`, `platformer`, etc.) contenga únicamente la lógica, arte y reglas específicas de ese juego.

## Estructura de Archivos del Engine ⚙️

La carpeta `engine/` es el corazón de todos tus proyectos. Contiene los siguientes módulos:

```
/engine/
|-- game_loop.py         # El esqueleto de cualquier juego.
|-- asset_manager.py     # La biblioteca central de todos los recursos.
|-- spritesheet.py       # La herramienta para leer JSON de spritesheets.
|-- game_object.py       # El "ADN" de todo lo que se ve y se mueve.
|-- ui.py                # Componentes para interfaces (escenas, botones).
|-- dialogue.py          # Gestor de mensajes y conexión con APIs de IA.
|-- physics_object.py    # Clase base para objetos con físicas (PyMunk).
|-- ... (y otros que hemos creado)
```

-----

## Descripción de Componentes

### `engine/game_loop.py`

  * **Propósito:** Proporciona la estructura fundamental de un juego en Pygame. Gestiona la ventana, el reloj, y el bucle principal que procesa eventos, actualizaciones y dibujado.
  * **Clase Principal:** `GameLoop`
  * **Uso:** La clase principal de tu juego (ej: `ArkanoidGame`) **debe heredar** de esta clase. Esto te obliga a organizar tu código en métodos específicos, haciendo todo más limpio.
  * **Métodos a Sobrescribir:**
      * `handle_specific_events(self, event)`: Para la lógica de input (teclado, ratón).
      * `update_game_logic(self)`: Para toda la lógica de movimiento, colisiones y reglas del juego.
      * `draw_game_elements(self)`: Para dibujar todo en pantalla.

### `engine/asset_manager.py`

  * **Propósito:** Actúa como una biblioteca central para todos tus recursos (assets). Evita tener que cargar archivos desde el disco en medio del juego y centraliza toda la gestión de rutas.
  * **Clase Principal:** `AssetManager`
  * **Uso:** Creas una única instancia de esta clase en tu juego. En una fase de carga inicial, le dices todos los spritesheets e imágenes que quieres cargar, dándoles un "apodo". Durante el juego, le pides los recursos por ese apodo.
  * **Métodos Clave:**
      * `load_spritesheet(name, json_path)`: Carga un JSON y su imagen asociada.
      * `load_image(name, image_path)`: Carga una imagen suelta.
      * `get_sprite(sheet_name, sprite_name)`: Obtiene una imagen específica de un spritesheet.
      * `get_image(name)`: Obtiene una imagen suelta.

### `engine/spritesheet.py`

  * **Propósito:** Es la herramienta que lee y entiende el formato de archivo `.json` generado por tu editor de spritesheets.
  * **Clase Principal:** `Spritesheet`
  * **Uso:** El `AssetManager` usa esta clase internamente. Rara vez necesitarás llamarla directamente desde tu juego. Su trabajo es "recortar" la imagen grande del spritesheet en las pequeñas imágenes individuales que define el JSON.

### `engine/game_object.py`

  * **Propósito:** Es la clase base para cualquier objeto visible en tu juego (jugador, enemigos, balas, etc.). Proporciona la funcionalidad básica de tener una imagen (`self.image`), una posición y tamaño (`self.rect`), y maneja la animación de forma automática.
  * **Clase Principal:** `GameObject`
  * **Uso:** Todas las entidades de tu juego (`Paddle`, `Ball`, `Brick`) **deben heredar** de esta clase. Al crear una instancia, le pasas las imágenes (frames) ya cargadas por el `AssetManager`.
  * **Métodos Clave:**
      * `update()`: Debes recordar llamar a `super().update()` en las clases hijas para que la animación funcione.

### `engine/ui.py`

  * **Propósito:** Contiene las clases base para construir interfaces de usuario y gestionar el flujo entre diferentes pantallas del juego.
  * **Clases Principales:** `Scene`, `Button`.
  * **Uso:** Se utiliza para crear menús, pantallas de "Game Over", etc. La clase principal de tu juego gestiona qué `Scene` está activa, y cada escena maneja sus propios `Button` y lógica de UI.

-----

## Guía de Uso: Creando un Juego (Ejemplo Arkanoid) 🎮

Así es como todos los componentes del motor se unen para crear un juego.

### Paso 1: Estructura de Carpetas

Crea una carpeta para tu juego (`arkanoid/`) junto a la del motor (`engine/`). Dentro, organiza tus archivos específicos.

### Paso 2: El Punto de Entrada (`main.py`)

Este archivo debe ser lo más simple posible. Su única misión es crear una instancia de tu clase de juego principal y ejecutarla.

```python
# arkanoid/main.py
from .game import ArkanoidGame

def main():
    juego = ArkanoidGame()
    juego.run()

if __name__ == '__main__':
    main()
```

### Paso 3: La Clase Principal del Juego (`game.py`)

Este es el director de orquesta de tu juego.

```python
# arkanoid/game.py
import os
from engine.game_loop import GameLoop # Hereda del motor
from engine.asset_manager import AssetManager # Usa el motor
from .scenes.menu_scene import MenuScene # Gestiona las escenas del juego

class ArkanoidGame: # NO hereda de GameLoop directamente, sino que gestiona las escenas
    def __init__(self):
        # ... inicializa pygame, pantalla, reloj ...
        
        # 1. Define las rutas y crea el Gestor de Assets
        self.assets = AssetManager()
        self.load_assets() # Llama al método que carga todo

        # 2. Crea y gestiona las escenas
        self.scenes = { 'menu': MenuScene(self), 'game': GameScene(self) }
        self.current_scene = self.scenes['menu']
    
    def load_assets(self):
        # 3. Este es tu "Panel de Control" de recursos.
        # Le dices al gestor qué cargar y con qué apodo.
        self.assets.load_spritesheet('main_ss', os.path.join(self.assets_dir, 'arkanoid.json'))
        self.assets.load_image('ball_img', os.path.join(self.assets_dir, 'ball.png'))

    def run(self):
        # 4. El bucle principal delega todo a la escena activa.
        while self.running:
            # ...
            self.current_scene.update()
            self.current_scene.draw(self.screen)
            # ...
```

### Paso 4: Las Entidades (`entities.py`) 🧱

Aquí defines los "actores" de tu juego.

```python
# arkanoid/entities.py
from engine.game_object import GameObject # Hereda del motor

class Paddle(GameObject):
    def __init__(self, x, y, frames):
        # Le pasas los frames ya cargados por el AssetManager
        super().__init__(x, y, frames)
        self.velocidad = 8

    def update(self, *args, **kwargs):
        super().update() # ¡Importante para las animaciones!
        # ... lógica de movimiento de la pala ...

class Ball(GameObject):
    # ...
```

### Paso 5: Las Escenas (`scenes/`)

Cada pantalla (menú, juego, game over) es una clase que hereda de `engine.ui.Scene`. La escena del juego, por ejemplo (`game_scene.py`), es la que contiene la lógica de colisiones, puntuación y creación de entidades usando los assets que le proporciona el gestor principal.

```python
# arkanoid/scenes/game_scene.py
from engine.ui import Scene

class GameScene(Scene):
    def __init__(self, game):
        super().__init__(game)
        # self.game da acceso a todo lo global (assets, pantalla, etc.)
        
        # Usa el gestor de assets para crear los objetos
        pala_img = self.game.assets.get_sprite('main_ss', 'paleta')
        self.pala = Paddle(400, 550, pala_img)
        # ... etc ...
```

